# 🧠 Bayesian Case–Control Analysis for Spatial or Single-Cell Data

## Overview
This notebook implements a **hierarchical Bayesian case–control model** (inspired by the EUCLID framework) using **JAX + NumPyro**.  
The goal is to detect **case–control differences** in gene, metabolite, or lipid expression across **cell types**, while accounting for **sample-level** and **section-level** variability.

It runs directly on a `scanpy.AnnData` object and produces posterior estimates of differential expression with Bayesian confidence intervals and false discovery rate (FDR) control.

---

## 1️⃣ Purpose
Conventional differential expression tests (e.g. t-test, Wilcoxon) ignore hierarchical dependencies such as multiple samples or sections per condition.  
This model explicitly incorporates these nested sources of variation:

- **Condition:** case vs. control  
- **Sample:** biological replicates  
- **Section:** spatial or technical replicates  
- **Group:** cell type or spatial cluster  

The model estimates **posterior mean shifts** (akin to log2 fold change) and **posterior probabilities** of up/down regulation per cell type.

---

## 2️⃣ Model Structure

Each observation \( y_i \) (expression of one feature) is modeled as:

\[
y_i = \text{section\_effect}_{k[i]} +
      \text{base\_group}_{g[i]} +
      \text{shift\_group}_{g[i]} \cdot \mathbb{1}[\text{condition}_i = 1] +
      \varepsilon_i
\]

Where:
- **base_group** → baseline mean per cell type  
- **shift_group** → condition-specific deviation (main effect of interest)  
- **section_effect** → random section-level term (absorbs batch effects)  
- **ε_i ∼ N(0, σ)** → observation noise  

Hierarchical priors link these effects across samples and sections to stabilize estimates.

---

## 3️⃣ Implementation Steps

1. **Build a tidy DataFrame**  
   Extract one feature (e.g. gene/lipid) and metadata (`condition`, `sample_id`, `cell_class`) from `AnnData`.

2. **Construct design matrices**  
   Encode categorical variables as integer indices and define mappings (section→sample, sample→condition).

3. **Define the hierarchical model**  
   Using NumPyro’s probabilistic API with group-, sample-, and section-level random effects.

4. **Fit via Stochastic Variational Inference (SVI)**  
   - Automatic Normal guide (`AutoNormal`)  
   - Adam optimizer (`optax`)  
   - Monitor convergence via ELBO printed every 200 steps  

5. **Extract posterior summaries**  
   - Posterior mean and SD per group  
   - Posterior probability of positive/negative shift  
   - Bayesian FDR (*q*-value)  
   - Significance flag (`selected_fdr_0.05`)  

6. **Batch over multiple features**  
   Fit several genes or metabolites in one call with `run_casecontrol_on_adata()`.

---

## 4️⃣ Output

Each feature yields a tidy results table:

| group | posterior_mean | posterior_sd | p(>0) | p(<0) | qvalue | selected_fdr_0.05 |
|--------|----------------|---------------|-------|-------|--------|--------------------|
| Astrocytes | 0.023 | 0.011 | 0.91 | 0.09 | 0.04 | ✅ |
| Microglia  | –0.121 | 0.019 | 0.01 | 0.99 | 0.001 | ✅ |
| OPCs | 0.007 | 0.014 | 0.65 | 0.35 | 0.22 | ❌ |

Interpretation:
- **posterior_mean** ≈ log2 fold change  
- **qvalue** = Bayesian FDR  
- **True** → significant difference at 5% FDR  

---

## 5️⃣ Example Usage

```python
features = ["Hif1a", "Ndufa4l2", "Ldha"]

results = run_casecontrol_on_adata(
    adata,
    features=features,
    condition_key="condition",
    sample_key="sample_id",
    section_key="sample_id",
    group_key="cell_class",
    num_steps=1500,
    lr=0.01,  # smaller for stability
)

In [1]:
import scanpy as sc
import numpy as np
import pandas as pd
from scipy.sparse import issparse

import scanpy as sc

import jax
import jax.numpy as jnp
import jax.nn as jnn
from jax.ops import segment_sum

import numpyro
import numpyro.distributions as dist
from numpyro.infer import SVI, Trace_ELBO, Predictive
from numpyro.infer.autoguide import AutoNormal
import optax


# -----------------------------
# 1) Build a tidy DataFrame from AnnData
# -----------------------------

def make_casecontrol_df(
    adata,
    feature: str,
    condition_key: str = "condition",
    sample_key: str = "sample_id",
    section_key: str = "sample_id",
    group_key: str = "cell_class",
) -> pd.DataFrame:
    """
    Extract a single feature (gene/lipid) and relevant obs metadata
    into a tidy DataFrame.

    Columns:
      - feature (one column)
      - Condition
      - Sample
      - SectionID
      - group
    """
    if feature not in adata.var_names:
        raise ValueError(f"{feature} not found in adata.var_names")

    for k in [condition_key, sample_key, section_key, group_key]:
        if k not in adata.obs.columns:
            raise ValueError(f"Obs column '{k}' is missing from adata.obs")

    # Robust extraction of the feature vector (handles dense & sparse)
    mat = adata[:, feature].X
    if issparse(mat):
        values = mat.toarray().ravel().astype(np.float32)
    else:
        values = np.asarray(mat).ravel().astype(np.float32)

    df = pd.DataFrame({feature: values}, index=adata.obs_names)
    df["Condition"] = adata.obs[condition_key].values
    df["Sample"] = adata.obs[sample_key].values
    df["SectionID"] = adata.obs[section_key].values
    df["group"] = adata.obs[group_key].values
    return df


# -----------------------------
# 2) Encode design matrices
# -----------------------------

def make_design_matrices(df: pd.DataFrame) -> dict:
    """
    From a tidy DF with columns:
      Condition, Sample, SectionID, group
    build integer codes and mappings needed by the model.
    """

    # Encode as categorical codes
    for col, code_name in [
        ("Condition", "condition_code"),
        ("Sample", "sample_code"),
        ("SectionID", "section_code"),
        ("group", "group_code"),
    ]:
        df[code_name] = df[col].astype("category").cat.codes.astype(int)

    # Map from section -> sample
    sec2samp = (
        df[["section_code", "sample_code"]]
        .drop_duplicates()
        .sort_values("section_code")
    )
    section2sample = sec2samp["sample_code"].values.astype(int)

    # Map from sample -> condition
    samp2cond = (
        df[["sample_code", "condition_code"]]
        .drop_duplicates()
        .sort_values("sample_code")
    )
    sample2condition = samp2cond["condition_code"].values.astype(int)

    # Basic counts
    n_groups = int(df["group_code"].max() + 1)
    n_samples = int(df["sample_code"].max() + 1)
    n_sections = int(df["section_code"].max() + 1)
    n_conditions = int(df["condition_code"].nunique())
    n_obs = int(df.shape[0])

    if n_conditions != 2:
        raise ValueError(
            f"Model is currently implemented for exactly 2 conditions, "
            f"but found {n_conditions} in 'Condition'."
        )

    design = dict(
        condition_code=df["condition_code"].values.astype(int),
        section_code=df["section_code"].values.astype(int),
        group_code=df["group_code"].values.astype(int),
        section2sample=section2sample,
        sample2condition=sample2condition,
        n_groups=n_groups,
        n_samples=n_samples,
        n_sections=n_sections,
        n_conditions=n_conditions,
        n_obs=n_obs,
    )
    return design


# -----------------------------
# 3) Hierarchical case–control model (JAX/Numpyro)
# -----------------------------

def hierarchical_casecontrol_model(
    condition_code,
    section_code,
    group_code,
    section2sample,
    sample2condition,
    n_groups: int,
    n_samples: int,
    n_sections: int,
    n_conditions: int,
    n_obs: int,
    y=None,
    supertype_prior_sd: float = 1.0,
    supertype_shift_prior_sd: float = 1.0,
    sample_prior_sd: float = 1.0,
    section_prior_sd: float = 5.0,
    obs_sd: float = 0.1,
):
    """
    Hierarchical Gaussian model:

    y_{i} = section_effect[section_i]
            + base_group[group_i]
            + shift_group[group_i] * I[condition_i == 1]
            + eps_i

    with:
      - group-level base & shift
      - sample-level random effects
      - section-level random effects (centered by condition to absorb batch)
    """

    # ------------------------
    # Group-level (e.g. cell_class)
    # ------------------------
    with numpyro.plate("group_plate", n_groups):
        base_group = numpyro.sample(
            "base_group",
            dist.Normal(0.0, supertype_prior_sd),
        )
        shift_group = numpyro.sample(
            "shift_group",
            dist.Normal(0.0, supertype_shift_prior_sd),
        )

    # ------------------------
    # Sample-level
    # ------------------------
    with numpyro.plate("sample_plate", n_samples):
        mu_sample = numpyro.sample(
            "mu_sample",
            dist.Normal(0.0, sample_prior_sd),
        )
        log_sigma_sample = numpyro.sample(
            "log_sigma_sample",
            dist.Normal(0.0, section_prior_sd),
        )
    sigma_sample = jnn.softplus(log_sigma_sample)

    # ------------------------
    # Section-level (non-centered)
    # ------------------------
    with numpyro.plate("section_plate", n_sections):
        eps_section = numpyro.sample("eps_section", dist.Normal(0.0, 1.0))
        mu_sec = mu_sample[section2sample]          # [n_sections]
        sigma_sec = sigma_sample[section2sample]    # [n_sections]
        section_raw = mu_sec + eps_section * sigma_sec

    # Center section-level effects by condition (to break identifiability)
    cond_per_section = sample2condition[section2sample]  # shape [n_sections]

    sum_by_cond = segment_sum(section_raw, cond_per_section, n_conditions)
    count_by_cond = segment_sum(
        jnp.ones_like(section_raw), cond_per_section, n_conditions
    )
    mean_by_cond = sum_by_cond / count_by_cond

    section_effect = section_raw - mean_by_cond[cond_per_section]

    # ------------------------
    # Linear predictor per observation i
    # ------------------------
    mu = (
        section_effect[section_code]
        + base_group[group_code]
        + jnp.where(condition_code == 1, shift_group[group_code], 0.0)
    )

    # ------------------------
    # Observation model
    # ------------------------
    with numpyro.plate("obs", n_obs):
        numpyro.sample("y", dist.Normal(mu, obs_sd), obs=y)


# -----------------------------
# 4) Fit model for a single feature
# -----------------------------

def fit_casecontrol_single(
    df: pd.DataFrame,
    feature: str,
    num_steps: int = 1500,
    lr: float = 0.03,
    seed: int = 0,
    priors: dict | None = None,
):
    """
    Fit the hierarchical case–control model to a single feature column in df.

    Returns:
      stats_df : DataFrame with posterior stats per group (e.g. cell_class)
      samples  : dict of posterior samples for 'shift_group'
      params   : learned variational parameters
    """
    if priors is None:
        priors = {}

    design = make_design_matrices(df)
    y = df[feature].values.astype(np.float32)

    # Convert design arrays to jnp
    cc = jnp.array(design["condition_code"])
    scode = jnp.array(design["section_code"])
    gcode = jnp.array(design["group_code"])
    s2s = jnp.array(design["section2sample"])
    s2c = jnp.array(design["sample2condition"])
    n_groups = design["n_groups"]
    n_samples = design["n_samples"]
    n_sections = design["n_sections"]
    n_conditions = design["n_conditions"]
    n_obs = design["n_obs"]

    # Wrap model so we can pass priors cleanly
    def model_wrapper(
        condition_code,
        section_code,
        group_code,
        section2sample,
        sample2condition,
        n_groups,
        n_samples,
        n_sections,
        n_conditions,
        n_obs,
        y=None,
    ):
        return hierarchical_casecontrol_model(
            condition_code=condition_code,
            section_code=section_code,
            group_code=group_code,
            section2sample=section2sample,
            sample2condition=sample2condition,
            n_groups=n_groups,
            n_samples=n_samples,
            n_sections=n_sections,
            n_conditions=n_conditions,
            n_obs=n_obs,
            y=y,
            **priors,
        )

    guide = AutoNormal(model_wrapper)
    optimizer = optax.adam(lr)
    svi = SVI(model_wrapper, guide, optimizer, loss=Trace_ELBO())

    rng_key = jax.random.PRNGKey(seed)
    svi_state = svi.init(
        rng_key,
        condition_code=cc,
        section_code=scode,
        group_code=gcode,
        section2sample=s2s,
        sample2condition=s2c,
        n_groups=n_groups,
        n_samples=n_samples,
        n_sections=n_sections,
        n_conditions=n_conditions,
        n_obs=n_obs,
        y=jnp.array(y),
    )

    for step in range(num_steps):
        svi_state, loss = svi.update(
            svi_state,
            condition_code=cc,
            section_code=scode,
            group_code=gcode,
            section2sample=s2s,
            sample2condition=s2c,
            n_groups=n_groups,
            n_samples=n_samples,
            n_sections=n_sections,
            n_conditions=n_conditions,
            n_obs=n_obs,
            y=jnp.array(y),
        )
        if (step + 1) % 200 == 0:
            print(f"  step {step+1}/{num_steps}, ELBO={loss:.3f}")

    params = svi.get_params(svi_state)

    # Posterior samples for shift_group (per group)
    predictive = Predictive(
        model_wrapper,
        guide=guide,
        params=params,
        num_samples=1000,
        return_sites=["shift_group"],
    )
    samples = predictive(
        rng_key,
        condition_code=cc,
        section_code=scode,
        group_code=gcode,
        section2sample=s2s,
        sample2condition=s2c,
        n_groups=n_groups,
        n_samples=n_samples,
        n_sections=n_sections,
        n_conditions=n_conditions,
        n_obs=n_obs,
        y=None,
    )

    shift_samples = np.array(samples["shift_group"])  # shape [num_draws, n_groups]

    # Basic posterior summary + Bayesian FDR
    loc = shift_samples.mean(axis=0)
    sd = shift_samples.std(axis=0)
    p_pos = (shift_samples > 0).mean(axis=0)
    p_neg = (shift_samples < 0).mean(axis=0)
    PEP = np.minimum(p_pos, p_neg)  # posterior error probability

    order = np.argsort(PEP)
    cum_PEP = np.cumsum(PEP[order]) / np.arange(1, len(PEP) + 1)
    qvalue = np.empty_like(PEP)
    qvalue[order] = cum_PEP

    # map group_code -> group name
    mapping = (
        df[["group", "group_code"]]
        .drop_duplicates()
        .sort_values("group_code")
    )
    group_names = mapping["group"].values

    stats_df = pd.DataFrame(
        {
            "posterior_mean": loc,
            "posterior_sd": sd,
            "p(>0)": p_pos,
            "p(<0)": p_neg,
            "PEP": PEP,
            "qvalue": qvalue,
            "selected_fdr_0.05": qvalue < 0.05,
        },
        index=group_names,
    ).sort_values("qvalue")

    return stats_df, samples, params


# -----------------------------
# 5) High-level: run on several features directly from AnnData
# -----------------------------

def run_casecontrol_on_adata(
    adata,
    features,
    condition_key="condition",
    sample_key="sample_id",
    section_key="sample_id",
    group_key="cell_class",
    num_steps=1500,
    lr=0.03,
    seed=0,
):
    """
    Convenience wrapper:
      - pulls data from AnnData
      - fits hierarchical case–control model for each feature
      - returns dict {feature -> stats_df}
    """
    results = {}
    for f in features:
        print(f"\n=== Fitting case–control model for {f} ===")
        df = make_casecontrol_df(
            adata,
            feature=f,
            condition_key=condition_key,
            sample_key=sample_key,
            section_key=section_key,
            group_key=group_key,
        )
        stats_f, _, _ = fit_casecontrol_single(
            df,
            feature=f,
            num_steps=num_steps,
            lr=lr,
            seed=seed,
        )
        results[f] = stats_f
    return results

/Users/christoffer/miniconda3/envs/EUCLID_ENV/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
print(jax.devices())

Platform 'METAL' is experimental and not all JAX functionality may be correctly supported!


Metal device set to: Apple M4 Max
[METAL(id=0)]


W0000 00:00:1762936099.901459 1729667 mps_client.cc:510] WARNING: JAX Apple GPU support is experimental and not all JAX functionality is correctly supported!
I0000 00:00:1762936099.917562 1729667 service.cc:145] XLA service 0x600002a0df00 initialized for platform METAL (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1762936099.917572 1729667 service.cc:153]   StreamExecutor device (0): Metal, <undefined>
I0000 00:00:1762936099.918648 1729667 mps_client.cc:406] Using Simple allocator.
I0000 00:00:1762936099.918661 1729667 mps_client.cc:384] XLA backend will use up to 38654230528 bytes on device 0 for SimpleAllocator.


# read data

In [3]:
adata = sc.read_h5ad('/Users/christoffer/work/karolinska/development/oligo-mtDSB/data/mtDNA_DSB_5k_clustered_annotation_with_rbd_2.h5ad')

In [4]:
adata.obs

,x_centroid,y_centroid,transcript_counts,control_probe_counts,genomic_control_counts,control_codeword_counts,unassigned_codeword_counts,deprecated_codeword_counts,total_counts,cell_area,...,rbd_domain_0.2,rbd_domain_0.5,rbd_domain_1.0,RBD_compartment,RBD_compartment_simplified,leiden_1.5,leiden_2.1,leiden_2.3,leiden_2.5,leiden_3
aaaaakjg-1,67.136543,1092.286377,631,0,0,0,1,0,631.0,37.434533,...,0,0,5,Cortex III,Cortex,21,27,25,27,29
aaaaikjn-1,80.635910,779.488647,1102,0,0,0,2,0,1102.0,90.538285,...,0,0,38,Unknown III,Unknown,3,29,27,32,30
aaaapoln-1,67.554489,1513.168579,335,0,0,0,0,0,335.0,20.817032,...,0,0,4,Cortex II,Cortex,9,14,13,10,10
aaacmkgf-1,70.575630,1138.702881,1931,0,0,0,1,0,1931.0,81.055472,...,0,0,5,Cortex III,Cortex,9,29,27,32,30
aaaefdnf-1,583.425537,1142.197876,1105,0,0,0,1,0,1105.0,68.276252,...,0,0,5,Cortex III,Cortex,17,17,22,17,17
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
oifndkbo-1,2379.717529,4785.681152,419,0,0,0,0,0,419.0,42.221095,...,1,2,0,Hypothalamus,Hypothalamus,4,21,37,42,45
oifolalk-1,2281.881104,4803.725098,219,0,0,0,0,0,219.0,36.170158,...,1,2,0,Hypothalamus,Hypothalamus,19,11,15,19,40
oifphdni-1,2343.403564,4830.252441,76,0,0,0,0,0,76.0,14.946719,...,1,2,0,Hypothalamus,Hypothalamus,4,21,37,38,33
oigabhlk-1,2341.813477,4843.651367,176,0,0,0,0,0,176.0,30.661095,...,1,2,0,Hypothalamus,Hypothalamus,19,11,15,19,40


In [64]:
sc.pp.highly_variable_genes(adata)

In [66]:
adata.var[adata.var.highly_variable].loc['Atf5']

n_cells_by_counts           146219
mean_counts               0.191538
pct_dropout_by_counts    85.134031
total_counts              188393.0
highly_variable               True
means                     0.022473
dispersions              -1.239583
dispersions_norm          0.695198
Name: Atf5, dtype: object

In [58]:
adata.var[adata.var.highly_variable].iloc['Serpina3n']

n_cells_by_counts            64140
mean_counts               0.096056
pct_dropout_by_counts    93.478937
total_counts               94479.0
highly_variable              False
means                     0.010318
dispersions              -1.004486
dispersions_norm          1.364394
Name: Serpina3n, dtype: object

In [71]:
features = adata.var_names[adata.var["highly_variable"]].tolist()

In [75]:
from pathlib import Path
os.chdir(Path().resolve().parent)

In [11]:
import os

# Set your desired working directory
new_dir = "/Users/christoffer/work/karolinska/development/oligo-mtDSB/notebooks-02"  # <-- change this path
os.chdir(new_dir)

In [81]:
import os
import sys
import pandas as pd

# ============
# Setup
# ============
outdir = "../results/casecontrol_hvg_results_live"

os.makedirs(outdir, exist_ok=True)

# --- Choose which features to run on ---
# Option 1: only highly variable genes
USE_HVGS = True   # set to False if you want all genes instead

# Option 2 (optional): custom list like `present`
USE_CUSTOM_LIST = False
CUSTOM_FEATURES = [
    # e.g. "Serpina3n", "Gfap", ...
]

if USE_CUSTOM_LIST:
    features = [g for g in CUSTOM_FEATURES if g in adata.var_names]
    missing = sorted(set(CUSTOM_FEATURES) - set(features))
    if missing:
        print(f"Warning: {len(missing)} custom features not found in var_names: {missing[:10]}")
elif USE_HVGS and "highly_variable" in adata.var.columns:
    features = adata.var_names[adata.var["highly_variable"]].tolist()
    features = features + [
        "Atf4", "Ddit3", "Hspa5", "Atf5",
        "Mtf1", "Mt1", "Mt2", "Sod2", "Hif1a",
        "Hk2", "Pfkl", "Ldha", "Slc16a1", "Slc16a3",
        "Tmem173", "Ifit1", "Ifit2", "Ifit3", "Isg15", "Rsad2",
        "Nfkb2", "Tnfaip3", "Ccl2",
        "Mbp", "Plp1", "Mog", "Serpina3n",'Atf4','Atf5'
    ]
else:
    # all genes in adata
    features = adata.var_names.tolist()

print(f"Running hierarchical case–control across {len(features)} features...")

# Track combined results incrementally
summary_path = os.path.join(outdir, "casecontrol_summary_live.csv")

# If resuming, load already processed genes
if os.path.exists(summary_path):
    done_genes = pd.read_csv(summary_path)["feature"].unique().tolist()
    print(f"Resuming: {len(done_genes)} already completed.")
else:
    done_genes = []

# ============
# Streaming run
# ============

for i, gene in enumerate(features, start=1):
    if gene in done_genes:
        print(f"[{i}/{len(features)}] Skipping {gene} (already done).", flush=True)
        continue

    print(f"\n=== [{i}/{len(features)}] Fitting case–control model for {gene} ===", flush=True)
    try:
        # Prepare data
        df_gene = make_casecontrol_df(
            adata,
            feature=gene,
            condition_key="condition",
            sample_key="sample_id",
            section_key="sample_id",
            group_key="cell_class",
        )

        # Fit model
        stats_gene, _, _ = fit_casecontrol_single(
            df_gene,
            feature=gene,
            num_steps=800,
            lr=0.03,
            seed=0,
        )

        stats_gene = stats_gene.copy()
        stats_gene["feature"] = gene

        # Save per-feature CSV
        gene_path = os.path.join(outdir, f"{gene}_casecontrol.csv")
        stats_gene.to_csv(gene_path)
        print(f"  ✅ Saved per-feature results → {gene_path}", flush=True)

        # Append to combined CSV incrementally
        header = not os.path.exists(summary_path)
        stats_gene.to_csv(summary_path, mode="a", header=header)
        print(f"  📎 Appended to {summary_path}", flush=True)

    except Exception as e:
        print(f"  ❌ Error fitting {gene}: {e}", flush=True)
        continue

print("\n🎉 All features processed (or skipped if already done).")

Running hierarchical case–control across 370 features...

=== [1/370] Fitting case–control model for Abca1 ===
  step 200/800, ELBO=-915947.688
  step 400/800, ELBO=-1088648.875
  step 600/800, ELBO=-1037941.812
  step 800/800, ELBO=-1192604.500
  ✅ Saved per-feature results → ../results/casecontrol_hvg_results_live/Abca1_casecontrol.csv
  📎 Appended to ../results/casecontrol_hvg_results_live/casecontrol_summary_live.csv

=== [2/370] Fitting case–control model for Abcb1a ===
  step 200/800, ELBO=-649555.375
  step 400/800, ELBO=-817060.750
  step 600/800, ELBO=-769557.312
  step 800/800, ELBO=-917848.688
  ✅ Saved per-feature results → ../results/casecontrol_hvg_results_live/Abcb1a_casecontrol.csv
  📎 Appended to ../results/casecontrol_hvg_results_live/casecontrol_summary_live.csv

=== [3/370] Fitting case–control model for Abcg2 ===
  step 200/800, ELBO=-829753.312
  step 400/800, ELBO=-999244.750
  step 600/800, ELBO=-950244.812
  step 800/800, ELBO=-1101540.125
  ✅ Saved per-feature

In [7]:
import os

In [10]:
!pwd

/Users/christoffer/work/karolinska/development/lipidomics-demyelination/notebooks


/Users/christoffer/miniconda3/envs/EUCLID_ENV/lib/python3.10/pty.py:89: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  pid, fd = os.forkpty()


In [12]:
base_dir = '../results/casecontrol_hvg_results_live_condition/'
df_ = []
files = os.listdir(base_dir)
for file in files:
    if file!='.ipynb_checkpoints':
        casecontrol = pd.read_csv(base_dir + file)
        df_.append(casecontrol)    

In [13]:
concat = pd.concat(df_)
concat = concat[concat['Unnamed: 0'] != 'Myocytes']

concat = concat.sort_values(by = 'posterior_mean', ascending = False)

In [16]:
GOI = []
for cell in concat['Unnamed: 0'].unique():
    df__ = concat[concat['Unnamed: 0'] == cell].sort_values(by = 'posterior_mean', ascending = False)
    df__ = df__[df__['selected_fdr_0.05']]
    GOI.append(list(df__['feature'].unique()))

In [17]:
GOI = [item for sublist in GOI for item in sublist]

In [18]:
GOI = np.unique(GOI)

In [21]:
GOI

array(['Adcy1', 'Adora2a', 'Agpat4', 'Ahi1', 'Aldoa', 'Aldoc', 'Anln',
       'Anxa2', 'Apod', 'Arc', 'Arrdc3', 'Aspa', 'Atf4', 'Atf5', 'Atp1b2',
       'B2m', 'Bcam', 'Capn2', 'Car2', 'Cav1', 'Ccnd2', 'Ccp110', 'Cd9',
       'Cldn10', 'Cldn11', 'Cnp', 'Cntn2', 'Cpox', 'Cx3cr1', 'Cxcl14',
       'Ddit4', 'Dusp6', 'Erbin', 'Ermn', 'Ezr', 'Fam107a', 'Fgfr3',
       'Gad2', 'Gfap', 'Gh', 'Gja1', 'Gjb6', 'Gng11', 'Gpc4', 'Gpr17',
       'Gpr37l1', 'Gpr88', 'Gprc5b', 'Grn', 'Gstp1', 'H2-D1', 'H2-K1',
       'Hexb', 'Hist1h2bc', 'Hspa5', 'Id4', 'Igf2', 'Ighm', 'Il33',
       'Itgb5', 'Itih5', 'Jun', 'Kdr', 'Ldha', 'Mag', 'Mal', 'Mbp',
       'Meg3', 'Meis2', 'Mertk', 'Mfsd2a', 'Mlc1', 'Mmp15', 'Mog', 'Mt2',
       'Nkx6-2', 'Nmu', 'Nsmf', 'Ntsr2', 'Olig2', 'P2ry12', 'Parvb',
       'Pde10a', 'Pdyn', 'Peg10', 'Plin4', 'Pllp', 'Plpp3', 'Pltp',
       'Ppp1r16b', 'Ppp1r1b', 'Prl', 'Prr5l', 'Ptgds', 'Qdpr', 'Rbp1',
       'Rgcc', 'Rgs9', 'Rnaset2a', 'Rnf13', 'S100a1', 'Scn4b', 'Selenop',
       

In [19]:
import jax
jax.devices()

[METAL(id=0)]

In [ ]:
import os
import pandas as pd

# ============
# Setup
# ============
outdir = "../results/casecontrol_by_age_results"
os.makedirs(outdir, exist_ok=True)

ages = ["21", "60"]  # adjust to your obs values
features = GOI

all_results = {}

print(f"Running hierarchical case–control across {len(features)} features × {len(ages)} ages...")

# ============
# Loop per age
# ============
for age in ages:
    print(f"\n=== Running for age: {age} ===", flush=True)
    ad_sub = adata[adata.obs["age"] == age].copy()
    
    try:
        # Fit model for all features in this age subset
        res = run_casecontrol_on_adata(
            ad_sub,
            features=features,
            condition_key="condition",   # mtDSB vs control
            sample_key="sample_id",
            section_key="sample_id",
            group_key="cell_class",
            num_steps=800, lr=0.03, seed=0
        )

        # Stack into one DataFrame
        df_age = (
            pd.concat({k: v.assign(feature=k) for k, v in res.items()})
              .reset_index(drop=True)
              .rename(columns={"index": "group"})
        )
        df_age["age"] = age
        all_results[age] = df_age

        # Save per-age results
        out_path = os.path.join(outdir, f"casecontrol_{age}.csv")
        df_age.to_csv(out_path, index=False)
        print(f"  ✅ Saved results for {age} → {out_path}", flush=True)

    except Exception as e:
        print(f"  ❌ Error at age {age}: {e}", flush=True)
        continue

# ============
# Combine into one table
# ============
cc = pd.concat(all_results.values(), ignore_index=True)
summary_path = os.path.join(outdir, "casecontrol_combined.csv")
cc.to_csv(summary_path, index=False)
print(f"\n📎 Combined summary saved to: {summary_path}")

Running hierarchical case–control across 126 features × 2 ages...

=== Running for age: 21 ===

=== Fitting case–control model for Adcy1 ===
  step 200/800, ELBO=-142366.250
  step 400/800, ELBO=-148242.062
  step 600/800, ELBO=-289358.906
  step 800/800, ELBO=-348043.281

=== Fitting case–control model for Adora2a ===
  step 200/800, ELBO=-335055.125
  step 400/800, ELBO=-340520.031
  step 600/800, ELBO=-479855.969
  step 800/800, ELBO=-540125.938

=== Fitting case–control model for Agpat4 ===
  step 200/800, ELBO=-215631.281
  step 400/800, ELBO=-217295.422
  step 600/800, ELBO=-357135.312
  step 800/800, ELBO=-416882.656

=== Fitting case–control model for Ahi1 ===
  step 200/800, ELBO=1798.520
  step 400/800, ELBO=-1509.877
  step 600/800, ELBO=-142218.016
  step 800/800, ELBO=-200962.219

=== Fitting case–control model for Aldoa ===
  step 200/800, ELBO=676536.188
  step 400/800, ELBO=668965.500
  step 600/800, ELBO=523114.625
  step 800/800, ELBO=470042.500

=== Fitting case–cont

In [1]:
sc.

SyntaxError: invalid syntax (3067123875.py, line 1)